# Stratification timing: runtime vs. dimension $d$

Runtime of **tensor stratification** on the SphereLab problem as a function of the axis
dimension $d$ of a cubical $d \times d \times d$ tensor with a hidden sphere.

The input is `labs/SphereLab.ipynb` section 3's construction at scale: axis values
$u_i = x_i^2 - r^2/3$ with the support where the three $u$'s sum to zero, so the sphere's
equation holds exactly on the lattice; then a random **orthogonal** change of basis on every
axis, and `nondeg`. That is what hides the sphere.

`stratify` is two phases, timed apart because only the first depends on the solver:

- **`der_seconds`** — `derTrOpsReduced`, the null-space solve.
- **`strat_seconds`** — real canonical form per axis, then act on $\Gamma$.

**The grid.** $d = 10, 15, 20, \ldots$; `Float32` and `Float64`; the universal chisel against
`UniversalOp()` and `SymmetricOp()`; the general method `SylverLining` against **every
registered null solver**, plus `Auto`, `QuickDer` against three of them, `QuickDer3` and
`SymmetricGram`.

**Tolerance.** Every run solves at `solve_tol = 1e-6` and is *scored* against an accuracy
target of `1e-8` (Float32) and `1e-16` (Float64). Solver tolerance and attained accuracy are
different things — see the accuracy figure at the end.

**Drop-out.** A configuration whose total time passes **60 s** at some $d$ is not run at any
larger $d$. Its line simply ends, and where it ends is the result.

In [ ]:
using Pkg
Pkg.activate("..")          # the OpenDleto project

using CSV, DataFrames, PlotlyJS

PlotlyJS.templates.default = "plotly_white"

# The sweep is long and APPENDS as it goes, so this notebook is meant to be
# re-run against a partial file.  Two things follow.
#
# 1. Fix the column types.  A header-only CSV -- the state between launching
#    the sweep and its first measured row -- gives every column the type
#    `Missing` if CSV.jl is left to infer, and the first `minimum(df.d)` then
#    fails with "reducing over an empty collection".  Declaring the schema
#    means an empty file still yields a correctly typed, zero-row frame.
# 2. Drop a trailing partial line.  The driver can be mid-write when this runs.
const COLTYPES = Dict(
    :d => Int, :valence => Int, :eltype => String, :solve_tol => Float64,
    :target => Float64, :ops => String, :method => String, :solver => String,
    :der_seconds => Float64, :strat_seconds => Float64, :total_seconds => Float64,
    :bytes => Float64, :nullity => Int, :residual => Float64,
    :meets_target => Bool, :lsq_err => Float64, :support => Float64,
    :perm_ok => Bool, :dims => String, :nnz => Int, :status => String)

function load_timing(path = "stratify-timing.csv")
    isfile(path) || error("$path not found. Run:  bench/jl timing/StratifyTimingMacStudio.jl 150 60")
    lines = readlines(path)
    isempty(lines) && error("$path is empty; the sweep has not started.")
    want = count(==(','), lines[1])
    keep = [lines[1]; filter(l -> count(==(','), l) == want, lines[2:end])]
    return CSV.read(IOBuffer(join(keep, "\n")), DataFrame; types = COLTYPES)
end

df = load_timing()
ok = filter(:status => ==("ok"), df)

if nrow(df) == 0
    @warn "No measured rows yet -- the sweep writes its header first and then " *
          "warms up every configuration before the first timing. The figures " *
          "below will render as placeholders until rows appear; re-run this " *
          "cell to pick up progress."
else
    println(nrow(df), " runs, ", nrow(df) - nrow(ok), " of them did not complete; ",
            "d = ", minimum(df.d), " to ", maximum(ok.d))
end

In [ ]:
# ---------------------------------------------------------------- the palette
#
# Two encodings are used, and each is validated for its own job.
#
# SMALL MULTIPLES.  One panel per configuration, so within a panel the only
# distinctions are precision (two colours) and operator space (solid vs dotted).
# Two slots clear every gate including 3:1 contrast, so those panels need no
# relief of any kind.
#
# THE LEADERS OVERLAY.  Up to six configurations share one panel, so it takes
# the first six slots of the validated categorical order (worst adjacent CVD
# ΔE 9.1, normal-vision ΔE 19.6 on the light surface).  Three of those sit
# under 3:1 contrast, so every line in those figures carries a direct end
# label -- that is the relief, and it is why there is no table anywhere here.

const PRECISION_COLOR = Dict("Float32" => "#2a78d6", "Float64" => "#eb6834")
const OPS_DASH        = Dict("universal" => "solid",  "symmetric" => "dot")

const LEADER_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]

const INK     = "#52514e"     # labels never wear the series colour
const MUTED   = "#b8b7b0"
const SURFACE = "#fcfcfb"

const PANELS = [("Float32", "universal", 1, 1), ("Float32", "symmetric", 1, 2),
                ("Float64", "universal", 2, 1), ("Float64", "symmetric", 2, 2)]
const PANEL_TITLES = ["Float32 · universal ops"  "Float32 · symmetric ops"
                      "Float64 · universal ops"  "Float64 · symmetric ops"]

rows(data, elt, o) = data[(data.eltype .== elt) .& (data.ops .== o), :]
series(data, m) = sort(data[data.method .== m, :], :d)

"""
    note(msg) -> Plot

A placeholder figure carrying `msg`.  Every figure cell below routes through
this when there is nothing to draw yet, so a partial sweep renders a readable
notebook instead of an error.
"""
note(msg) = Plot(scatter(x = Float64[], y = Float64[], showlegend = false),
                 Layout(height = 160, width = 1080,
                        xaxis = attr(visible = false), yaxis = attr(visible = false),
                        margin = attr(l = 20, r = 20, t = 20, b = 20),
                        annotations = [attr(text = msg, showarrow = false,
                                            xref = "paper", yref = "paper",
                                            x = 0.5, y = 0.5,
                                            font = attr(color = INK, size = 14))]))

"Set every y axis of an n-panel subplot figure to a log scale."
function logy!(p, n)
    kw = Dict{Symbol,Any}()
    for i in 1:n
        kw[Symbol("yaxis", i == 1 ? "" : i, "_type")] = "log"
    end
    relayout!(p; kw...)
end
nothing

## Every configuration, one panel each

One small panel per configuration so that all of them are legible at once without asking a
legend to carry more identities than colour can hold. Inside a panel, **colour is the
precision** and **dash is the operator space**. Panels are ordered by how far the
configuration got: the ones that reach the largest $d$ come first, and a panel whose lines
stop early is a configuration that hit the 60 s budget there.

Shared axes throughout, so panels are directly comparable by eye.

In [ ]:
"""
    all_configs_figure(ok)

Small multiples: one panel per configuration, ordered by how far it got.
Colour is the precision, dash is the operator space.
"""
function all_configs_figure(ok)
    reach = sort(combine(groupby(ok, :method), :d => maximum => :dmax, :total_seconds => maximum => :tmax),
                 [:dmax, :tmax], rev = [true, false])
    methods_by_reach = reach.method

    NC = 4
    NR = ceil(Int, length(methods_by_reach) / NC)
    titles = fill("", NR, NC)
    for (k, m) in enumerate(methods_by_reach)
        titles[fld(k - 1, NC) + 1, mod(k - 1, NC) + 1] = m
    end

    p = make_subplots(rows = NR, cols = NC, subplot_titles = titles,
                      shared_xaxes = true, shared_yaxes = true,
                      vertical_spacing = 0.035, horizontal_spacing = 0.035)
    seen = Set{String}()
    for (k, m) in enumerate(methods_by_reach)
        r, c = fld(k - 1, NC) + 1, mod(k - 1, NC) + 1
        for (elt, o, _, _) in PANELS
            s = series(rows(ok, elt, o), m)
            nrow(s) == 0 && continue
            key = elt * "/" * o
            show_it = !(key in seen); push!(seen, key)
            add_trace!(p, scatter(
                x = s.d, y = s.total_seconds, mode = "lines+markers",
                name = "$elt · $o ops", legendgroup = key, showlegend = show_it,
                line = attr(color = PRECISION_COLOR[elt], width = 2, dash = OPS_DASH[o]),
                marker = attr(color = PRECISION_COLOR[elt], size = 6,
                              line = attr(color = SURFACE, width = 1)),
                hovertemplate = "d = %{x}<br>%{y:.3g} s<extra>" * m * "<br>" * key * "</extra>",
            ), row = r, col = c)
        end
    end
    logy!(p, NR * NC)
    relayout!(p, title_text = "Every configuration: total stratification time vs. d (log scale, shared axes)",
              height = 240 * NR + 170, width = 1080,
              legend = attr(orientation = "h", y = -0.06, x = 0),
              margin = attr(l = 70, r = 30, t = 110, b = 90))
    p
end

isempty(ok) ? note("no measured rows yet — re-run the loading cell once the sweep has written some") :
              all_configs_figure(ok)

## The leaders

The six configurations that reach the largest $d$ in each panel, overlaid so they can be
compared directly. Each line is labelled at its last measured point, which is where it hit
the 60 s budget — so the labels sit at different $x$ and do not collide.

In [ ]:
"""
    leaders(data, elt, o; n = 6)

The `n` configurations in this panel that reach the largest d, tie-broken by being faster
at that d.
"""
function leaders(data, elt, o; n = 6)
    sub = rows(data, elt, o)
    isempty(sub) && return String[]
    g = combine(groupby(sub, :method), :d => maximum => :dmax,
                :total_seconds => (v -> minimum(v)) => :best)
    first(sort(g, [:dmax, :best], rev = [true, false]).method, min(n, nrow(g)))
end

"""
    leader_figure(data, ycol; title, ylab, logscale = true, hline = nothing)

A 2x2 facet -- precision down, operator space across -- showing only that panel's leaders,
one colour each, every line labelled at its last point.  `hline` draws reference rules at
the given (y, text) pairs.
"""
function leader_figure(data, ycol; title, ylab, logscale = true, hline = nothing)
    p = make_subplots(rows = 2, cols = 2, subplot_titles = PANEL_TITLES,
                      shared_xaxes = true, vertical_spacing = 0.12,
                      horizontal_spacing = 0.07)
    for (elt, o, r, c) in PANELS
        sub = rows(data, elt, o)
        for (k, m) in enumerate(leaders(data, elt, o))
            s = series(sub, m)
            nrow(s) == 0 && continue
            col = LEADER_COLORS[mod1(k, length(LEADER_COLORS))]
            add_trace!(p, scatter(
                x = s.d, y = s[!, ycol], mode = "lines+markers+text",
                name = m, showlegend = false,
                line = attr(color = col, width = 2),
                marker = attr(color = col, size = 8, line = attr(color = SURFACE, width = 2)),
                text = [i == nrow(s) ? " " * m : "" for i in 1:nrow(s)],
                textposition = "middle right", textfont = attr(color = INK, size = 10),
                hovertemplate = "d = %{x}<br>%{y:.4g}<extra>" * m * "</extra>",
            ), row = r, col = c)
        end
        if hline !== nothing
            # The rule spans this panel's own measured range, not the frame's.
            ext = isempty(sub) ? (minimum(data.d), maximum(data.d)) :
                                 (minimum(sub.d), maximum(sub.d))
            for (yv, txt) in hline
                add_trace!(p, scatter(x = [ext[1], ext[2]], y = [yv, yv],
                    mode = "lines", showlegend = false, hoverinfo = "skip",
                    line = attr(color = MUTED, width = 1, dash = "dash")), row = r, col = c)
            end
        end
    end
    logscale && logy!(p, 4)
    relayout!(p, title_text = title, height = 800, width = 1080, hovermode = "closest",
              margin = attr(l = 80, r = 190, t = 100, b = 80),
              xaxis3_title_text = "axis dimension d", xaxis4_title_text = "axis dimension d",
              yaxis_title_text = ylab, yaxis3_title_text = ylab)
    return p
end

isempty(ok) ? note("no measured rows yet") :
              leader_figure(ok, :total_seconds;
                            title = "The leaders: total stratification time vs. d (log scale, 60 s budget)",
                            ylab = "seconds")

## Where the time goes

Solid is the **derivation solve**, dotted is the **stratification** that follows it, on one
shared log axis so the gap between them is the real ratio. The stratification is an
eigendecomposition per axis plus three contractions; it does not depend on the solver, which
is why the dotted lines lie on top of one another inside a panel.

In [ ]:
"Solid: the derivation solve.  Dotted: the stratification that follows it."
function split_figure(ok)
    p = make_subplots(rows = 2, cols = 2, subplot_titles = PANEL_TITLES,
                      shared_xaxes = true, vertical_spacing = 0.12, horizontal_spacing = 0.07)
    for (elt, o, r, c) in PANELS
        sub = rows(ok, elt, o)
        for (k, m) in enumerate(leaders(ok, elt, o))
            s = series(sub, m)
            nrow(s) == 0 && continue
            col = LEADER_COLORS[mod1(k, length(LEADER_COLORS))]
            add_trace!(p, scatter(x = s.d, y = s.der_seconds, mode = "lines+markers+text",
                name = m, showlegend = false,
                line = attr(color = col, width = 2),
                marker = attr(color = col, size = 8, line = attr(color = SURFACE, width = 2)),
                text = [i == nrow(s) ? " " * m : "" for i in 1:nrow(s)],
                textposition = "middle right", textfont = attr(color = INK, size = 10),
                hovertemplate = "d = %{x}<br>derivation %{y:.4g} s<extra>" * m * "</extra>",
            ), row = r, col = c)
            add_trace!(p, scatter(x = s.d, y = s.strat_seconds, mode = "lines",
                showlegend = false,
                line = attr(color = col, width = 2, dash = "dot"),
                hovertemplate = "d = %{x}<br>stratification %{y:.4g} s<extra>" * m * "</extra>",
            ), row = r, col = c)
        end
    end
    logy!(p, 4)
    relayout!(p, title_text = "Solid: the derivation solve.  Dotted: the stratification that follows it.",
              height = 800, width = 1080, hovermode = "closest",
              margin = attr(l = 80, r = 190, t = 100, b = 80),
              xaxis3_title_text = "axis dimension d", xaxis4_title_text = "axis dimension d",
              yaxis_title_text = "seconds", yaxis3_title_text = "seconds")
    p
end

isempty(ok) ? note("no measured rows yet") : split_figure(ok)

In [ ]:
if !isempty(ok)
    ok.der_frac = ok.der_seconds ./ ok.total_seconds
end

isempty(ok) ? note("no measured rows yet") :
              leader_figure(ok, :der_frac;
                            title = "Fraction of the run spent solving for the derivation (1.0 = the solve is everything)",
                            ylab = "der / total", logscale = false)

## Did it solve the problem, and to what accuracy?

A timing figure is only worth reading if the runs it times found the sphere. Two checks,
both as lines.

**Accuracy attained.** `residual` is the Z-law residual `Dleto.der_residual` of the
derivation each run stratified along — the defining equation, relative to the size of the
data. The dashed rules mark the `1e-8` and `1e-16` targets.

In [ ]:
isempty(ok) ? note("no measured rows yet") :
              leader_figure(ok, :residual;
                            title = "Accuracy attained: Z-law residual of the derivation (dashed rules: 1e-8 and 1e-16 targets)",
                            ylab = "relative residual",
                            hline = [(1e-8, "1e-8"), (1e-16, "1e-16")])

**Nullity found.** Under symmetric operators the derivation space of this input is exactly
three-dimensional — the two scalar derivations plus the sphere's — so a line sitting on 3 is
a run that found the sphere and nothing spurious. A line that drops to 0 is a solver that
found no derivation at all; a line above 3 under universal operators is the unrestricted
chisel admitting more derivations than the sphere's, which is a property of the input, not a
solver failure.

In [ ]:
isempty(ok) ? note("no measured rows yet") :
              leader_figure(ok, :nullity;
                            title = "Derivation space found (3 = the sphere, under symmetric operators)",
                            ylab = "nullity", logscale = false, hline = [(3.0, "3")])

## Reproducing this

From the repository root, with the project's Julia wrapper — `bench/jl` holds the thread,
heap and RSS budget for a shared machine, so never invoke bare `julia` here:

```bash
bench/jl timing/StratifyTiming.jl 150 60      # maxd 150, 60 s budget
```

The driver is [`StratifyTiming.jl`](StratifyTiming.jl); it runs `stratify(Ω, ch, Γ)`'s own
body unrolled so the two timed phases are the real ones. Input construction and the
reconstruction score come from `bench/SphereHarness.jl`. Delete `timing/stratify-timing.csv`
first for a clean sweep — the driver appends.

The driver **refuses to start without Arpack**. `:ArpackSolver` lives behind a weak
dependency, and `:AutoSolver` chooses from the solvers actually registered — so a script that
forgets `using Arpack` does not lose a row, it silently measures a fall-through to a solver
this repo has clocked at 7–20× slower. That failure is what an earlier version of this
benchmark recorded, kept as `stratify-timing-literal-tol.csv`.